# Subset the results from estimating the parameters of the T cell model to those that follow our constraints

This file loads in the results of the exhaustive parameter sweep of $m$, $d$, $a$, $f$, and $l$, and selects the points that a) have the subline dominating at the end conform to the VAF experimental results and b) minimize error.

## Prep

Load needed packages

In [ ]:
%matplotlib widget
import matplotlib as mpl
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np 
import math
import seaborn as sns
import os
import time
from sklearn.linear_model import LinearRegression
from estimator import Estimator
from statsmodels.formula import api as smf
from scipy.stats import ttest_ind

Close any figures generated from previous runs

In [ ]:
plt.close("all")

## Define data and user-input parameters

Define the file path for the configuration file, the path to the saved parameter sweep estimates, and the file in which to save the subsetted results, and the names of the B6 groups.

In [ ]:
groups = ["Grp. A1 B6 (100% C1)", "Grp. A2 B6 (80% C1; 20% C11)", "Grp. A3 B6 (50% C1; 50% C11)", "Grp. A4 B6 (20% C1; 80% C11)"]
config = "config_tcell.json"
results_dir = "results_dir/"
data_save_file = "result_avg.csv"
fig_save_file = "figures/mda.svg" # Set to None if you don't want to save

Create an Estimator object from the config file.

In [ ]:
es = Estimator(config)

## Load data

Loop through the results files and create a data frame of all.

In [ ]:
results_raw = []
for fname in [f for f in os.listdir(results_dir) if (".csv" in f)]:
    if os.path.getsize("{}/{}".format(results_dir, fname)):
        temp = pd.read_csv("{}/{}".format(results_dir, fname))
        results_raw += [temp]
results_raw = pd.concat(results_raw, ignore_index=True).drop_duplicates()
results_raw

## Subset to parameter sets that satisfy constraints

Remove the parameter pairs that do not fulfill constraints for some run.

In [ ]:
# Parameter sets that have infinity error (this is returned when the wrong subline is dominating at the end point)
bad_pairs = results_raw[results_raw["error"] == np.inf]
# Merge the results with the set of bad parameter sets
results = pd.merge(results_raw, bad_pairs, on=["m", "d", "a", "f", "l"], how="left", indicator=True)
# Subset to the parameter sets that are not in bad_pairs
results = results[results["_merge"] == "left_only"]
# Drop unneeded columns and rename
results = results.drop(["group_y", "id_y", "k_y", "k_x", "winner_y", "error_y", "_merge"], axis=1)
results = results.rename({"group_x": "group", "id_x": "id", "winner_x": "winner", "error_x": "error"}, axis=1)
results

Subset the results to those where all mice have mean squared error in the top 50th percentile 

In [ ]:
# Make a copy of the results data frame so we maintain the original
temp1 = results.copy()
# For each mouse, determine which pairs have error in the top 50th percentile of errors (of that mouse)
temp1["top_quantile"] = results.groupby("id")["error"].transform(lambda x: x < np.quantile(x, 0.5))
# For each k, m pair, determine if every mouse has error in the top 50th percentile
temp1 = temp1.groupby(["m", "d", "a", "f", "l"]).apply(lambda x: all(x["top_quantile"])).reset_index()
# Rename column for ease of reading
temp1 = temp1.rename({0: "top_quantile"}, axis=1)
# Calculate the average values of each parameter combination 
temp2 = results.groupby(["m", "d", "a", "f", "l"])["error"].mean().reset_index()
# Merge the average error data frame and that containing the percentile information
avg_results = pd.merge(temp1, temp2, on=["m", "d", "a", "f", "l"])
avg_results

Add a column for $k$ based on the equation used (we deleted it earlier because of floating point issues).
Add a column for the a/f ratio.

In [ ]:
avg_results["k"] = es.km_slope*avg_results["m"] + es.km_intercept
avg_results["af"] = avg_results["a"]/avg_results["f"]
avg_results

Save newly subsetted data

In [ ]:
if data_save_file:
    avg_results.to_csv(data_save_file)

## Linear regression analysis

Perform linear regression between all pairs of variables

In [ ]:
lr_res = smf.ols("d~m", data=avg_results).fit()
print("d = {} * m + {}\tR^2 = {}".format(round(lr_res.params["m"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("a~m", data=avg_results).fit()
print("a = {} * m + {}\tR^2 = {}".format(round(lr_res.params["m"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("f~m", data=avg_results).fit()
print("f = {} * m + {}\tR^2 = {}".format(round(lr_res.params["m"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("l~m", data=avg_results).fit()
print("l = {} * m + {}\tR^2 = {}".format(round(lr_res.params["m"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("a~d", data=avg_results).fit()
print("a = {} * d + {}\tR^2 = {}".format(round(lr_res.params["d"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("f~d", data=avg_results).fit()
print("f = {} * d + {}\tR^2 = {}".format(round(lr_res.params["d"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("l~d", data=avg_results).fit()
print("l = {} * d + {}\tR^2 = {}".format(round(lr_res.params["d"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("f~a", data=avg_results).fit()
print("f = {} * a + {}\tR^2 = {}".format(round(lr_res.params["a"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("l~a", data=avg_results).fit()
print("l = {} * a + {}\tR^2 = {}".format(round(lr_res.params["a"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))
lr_res = smf.ols("l~f", data=avg_results).fit()
print("l = {} * f + {}\tR^2 = {}".format(round(lr_res.params["f"], 3), round(lr_res.params["Intercept"], 3), round(lr_res.rsquared, 3)))

## Plot data by error

Plot parameter points that satisfy constraints colored by average mean squared error

In [ ]:
# Plot data
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1, projection="3d")
pl = ax.scatter(avg_results["m"], avg_results["a"], avg_results["d"], c=avg_results["error"], s=6)
plt.colorbar(pl, ax=ax, fraction=0.04)
ax.set(xlabel="m", ylabel="a", zlabel="d")

# Set view
ax.view_init(elev=5, azim=83, roll=0)
plt.subplots_adjust(wspace=0.2, hspace=0.2)
plt.gca().invert_zaxis()

plt.tight_layout()
plt.show()

### Determine the groups

In [ ]:
avg_results.loc[(avg_results["m"] >= -0.09) & (avg_results["d"]==-1), "group"] = 1
avg_results.loc[(avg_results["m"] >= -0.09) & (avg_results["d"]==-5), "group"] = 2
avg_results.loc[(avg_results["group"].isna()) & (avg_results["d"] < 137*avg_results["m"] + 7), "group"] = 3
avg_results.loc[(avg_results["group"].isna()) & (avg_results["d"] >= 137*avg_results["m"] + 7), "group"] = 4

## Plot data by group

Plot parameter points that satisfy constraints colored by group

In [ ]:
# Plot data
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1, projection="3d")
pl = ax.scatter(avg_results["m"], avg_results["a"], avg_results["d"], c=avg_results["group"], s=2)
ax.set(xlabel="m", ylabel="a", zlabel="d")

# Add color bar
cmap = mpl.cm.viridis
bounds = [1, 2, 3, 4]
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend="min")
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.04, label="Group")
cbar.ax.set_yticklabels(["1a", "1b", "2a", "2b"])

# Set view
ax.view_init(elev=5, azim=83, roll=0)
plt.subplots_adjust(wspace=0.2, hspace=0.2)
plt.gca().invert_zaxis()

plt.tight_layout()

if fig_save_file:
    plt.savefig(fig_save_file)

plt.show()

## Analyze groups

Get the average values of the parameters for each group

In [ ]:
avg_results.groupby("group").mean()